# Задача 06. Когортный анализ удержания

**Постановка задачи:** нужно посмотреть, возвращаются ли клиенты после регистрации. Для каждой когорты регистрации посчитать, сколько клиентов сделали доставленный заказ в месяц регистрации, через 1 месяц, через 2 месяца и так далее.

Когорта — месяц `signup_date`. Активность — наличие доставленного заказа в соответствующий месяц.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
WITH customer_cohorts AS (
    SELECT
        customer_id,
        strftime('%Y-%m', signup_date) AS cohort_month
    FROM customers
), customer_activity AS (
    SELECT DISTINCT
        o.customer_id,
        strftime('%Y-%m', o.order_date) AS activity_month
    FROM orders o
    WHERE o.status = 'delivered'
), cohort_activity AS (
    SELECT
        cc.cohort_month,
        ca.activity_month,
        (
            (CAST(strftime('%Y', ca.activity_month || '-01') AS INTEGER) - CAST(strftime('%Y', cc.cohort_month || '-01') AS INTEGER)) * 12
            +
            (CAST(strftime('%m', ca.activity_month || '-01') AS INTEGER) - CAST(strftime('%m', cc.cohort_month || '-01') AS INTEGER))
        ) AS month_number,
        cc.customer_id
    FROM customer_cohorts cc
    JOIN customer_activity ca ON cc.customer_id = ca.customer_id
    WHERE ca.activity_month >= cc.cohort_month
), cohort_size AS (
    SELECT
        cohort_month,
        COUNT(customer_id) AS cohort_customers
    FROM customer_cohorts
    GROUP BY cohort_month
)
SELECT
    ca.cohort_month,
    ca.month_number,
    cs.cohort_customers,
    COUNT(DISTINCT ca.customer_id) AS active_customers,
    ROUND(100.0 * COUNT(DISTINCT ca.customer_id) / cs.cohort_customers, 2) AS retention_pct
FROM cohort_activity ca
JOIN cohort_size cs ON ca.cohort_month = cs.cohort_month
WHERE ca.month_number BETWEEN 0 AND 6
GROUP BY ca.cohort_month, ca.month_number, cs.cohort_customers
ORDER BY ca.cohort_month, ca.month_number;
"""

df = pd.read_sql_query(query, conn)
df

In [ ]:
# Сводная таблица удобнее для просмотра когорт
cohort_pivot = df.pivot(index='cohort_month', columns='month_number', values='retention_pct').fillna(0)
cohort_pivot.head(12)

**Комментарий стажёра:** в SQL получилась длинная таблица `cohort_month × month_number`. Для чтения я дополнительно развернул её в матрицу через pandas.